
# Hiberno-English Parser Evaluation Workflow

A clean, reproducible notebook for comparing:
- a **rule-based parser**
- a **GenAI / LLM parser**

This notebook is designed for:
- robust normalization
- safe merge/alignment
- field-level comparison
- containment-based groundedness diagnostics
- export of analysis-ready outputs

## Design principles

- Keep **raw values** and **normalized values** separate.
- Normalize **only what is necessary** for comparison.
- Treat **closed fields** (e.g. POS, regions) differently from open text fields.
- Prefer **small pure helper functions**.
- Make every major step **inspectable and exportable**.



## 1. Setup



In [42]:

from __future__ import annotations

import json
import math
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
from dataclasses import asdict, is_dataclass
from collections import Counter

import numpy as np
import pandas as pd
from pandas import json_normalize


pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)


In [43]:

# =========================
# CONFIG
# =========================

PROJECT_DIR = Path(".").resolve()

RULE_PATH = PROJECT_DIR / "Resources" / "rule_parser_output.jsonl"
GENAI_PATH = PROJECT_DIR / "Resources" / "genai_parser_output.jsonl"

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POS_MAP_PATH = PROJECT_DIR / "pos_map.json"
REGION_MAP_PATH = PROJECT_DIR / "region_map.json"

MERGED_JSONL_PATH = OUTPUT_DIR / "final_merged_normalized.jsonl"
MERGED_XLSX_PATH = OUTPUT_DIR / "final_merged_normalized.xlsx"
PRESENCE_SUMMARY_PATH = OUTPUT_DIR / "presence_summary.xlsx"
CONTAINMENT_REPORT_PATH = OUTPUT_DIR / "containment_failures.xlsx"

# Comparison fields
TEXT_FIELDS = [
    "headword",
    "headword_raw",
    "definition",
    "etymology",
]

LIST_FIELDS = [
    "variant_forms",
    "pronunciations",
    "part_of_speech",
    "examples",
    "cross_references",
    "region_mentions",
]

CORE_FIELDS = [
    "entry_id",
    "source_text",
    "headword_raw",
    "headword",
    "variant_forms_raw",
    "variant_forms",
    "pronunciations",
    "part_of_speech",
    "definition",
    "examples",
    "etymology",
    "cross_references",
    "region_mentions",
    "needs_review",
]



## 2. Helper functions

These helpers are intentionally small and pure.


In [44]:

# =========================
# GENERIC HELPERS
# =========================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)



def make_json_serializable(value):
    if is_dataclass(value):
        return asdict(value)

    if isinstance(value, dict):
        return {k: make_json_serializable(v) for k, v in value.items()}

    if isinstance(value, list):
        return [make_json_serializable(v) for v in value]

    if isinstance(value, tuple):
        return [make_json_serializable(v) for v in value]

    if pd.isna(value) and not isinstance(value, (list, dict, str)):
        return None

    return value


def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    records = df.to_dict(orient="records")
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            clean_record = {k: make_json_serializable(v) for k, v in record.items()}
            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")


def is_nullish(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    if isinstance(value, (list, tuple, set, dict)) and len(value) == 0:
        return True
    return False


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text)


def clean_text(value: Any) -> str | None:
    if is_nullish(value):
        return None
    text = str(value)
    text = normalize_unicode(text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def unique_keep_order(items: Iterable[Any]) -> list[Any]:
    seen = set()
    out = []
    for item in items:
        key = json.dumps(item, ensure_ascii=False, sort_keys=True) if isinstance(item, (dict, list)) else item
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def ensure_list(value: Any) -> list[Any]:
    if is_nullish(value):
        return []
    if isinstance(value, list):
        return value
    return [value]


def clean_list_text(value: Any) -> list[str]:
    cleaned = [clean_text(x) for x in ensure_list(value)]
    return unique_keep_order([x for x in cleaned if x is not None])


def safe_int_from_id(value: Any) -> int | None:
    if is_nullish(value):
        return None
    text = str(value).strip()
    match = re.search(r"(\d+)$", text)
    if match:
        return int(match.group(1))
    try:
        return int(text)
    except ValueError:
        return None


def char_len(value: Any) -> int:
    if is_nullish(value):
        return 0
    if isinstance(value, str):
        return len(value)
    if isinstance(value, list):
        return sum(char_len(v) for v in value)
    if isinstance(value, dict):
        return sum(char_len(v) for v in value.values())
    return len(str(value))


def word_len(value: Any) -> int:
    if is_nullish(value):
        return 0
    if isinstance(value, str):
        return len(re.findall(r"\S+", value))
    if isinstance(value, list):
        return sum(word_len(v) for v in value)
    if isinstance(value, dict):
        return sum(word_len(v) for v in value.values())
    return len(re.findall(r"\S+", str(value)))


In [45]:

# =========================
# TEXT COMPARISON HELPERS
# =========================

def normalize_text_for_match(value: Any) -> str:
    text = clean_text(value) or ""
    text = text.casefold()
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def to_text_list(value: Any) -> list[str]:
    if is_nullish(value):
        return []
    if isinstance(value, list):
        parts = [normalize_text_for_match(v) for v in value]
    elif isinstance(value, dict):
        parts = [normalize_text_for_match(v) for v in value.values()]
    else:
        parts = [normalize_text_for_match(value)]
    return [p for p in parts if p]


def exact_list_match(left: Any, right: Any) -> bool:
    return clean_list_text(left) == clean_list_text(right)


def exact_text_match(left: Any, right: Any) -> bool:
    return normalize_text_for_match(left) == normalize_text_for_match(right)



## 3. Load raw datasets


In [46]:

rule_raw = read_jsonl(RULE_PATH)
genai_raw = read_jsonl(GENAI_PATH)

print("rule_raw shape:", rule_raw.shape)
print("genai_raw shape:", genai_raw.shape)

display(rule_raw.head(2))
display(genai_raw.head(2))

#normalize para ID need to +1 to each gen AI outputs to match rule based (due to parsing differences)

genai_raw["entry_id"] = genai_raw["entry_id"].apply(lambda x: safe_int_from_id(x) + 1 if not is_nullish(x) else None)


rule_raw shape: (3898, 23)
genai_raw shape: (3899, 5)


,para_id,id,source_text,source_hash,parse_hash,entry_type,headword_raw,headword,variant_forms_raw,variant_forms,pronunciation,pronunciation_2,pronunciation_3,pronunciation_4,part_of_speech,grammatical_labels,definition,etymology,cross_references,examples,region_mentions,parse_confidence,parse_notes
0,1,hde_00001,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",56cda7efdbbbb923cd21f9621a720ed7276a6afde4c0bb100c68c681bc0aae9a,f035b1ef349b71d2273d1f2b7f707199a15277e81679def519d9e8bed9b5ab4c,lexical,a,a,NaN,[],ə,NaN,NaN,None,NaN,[],"indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’; ...",NaN,[],[She cried her ’nough.],"[{'code': 'PR', 'place': 'Mayo'}]",0.72,[POS not confidently parsed]
1,2,hde_00002,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",4d541985e55711105558ab10bc8e4c65b351fb5883ed961a1c428eb7eb0f362b,2f9068bf3a5bac4132726198ab606b31c06f464d9ad7a6f229bfcaa21006337a,lexical,a,a,NaN,[],ə,NaN,NaN,None,voc.,[],"particle (used when speaking to somebody). “Arrah, Tim, avourneen, why did you die?” Joyce, Finnegans Wake, 468.34: “I’m dreaming of ye, azores [a stór]”; Healy, Nineteen Acres, 9: “Run down to th...","Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘avourneen’, a term of endearment < Ir a mhuirnín. ‘Finneg...",[],"[Arrah, Tim, avourneen, why did you die?, I’m dreaming of ye, azores [a stór], Run down to the well, agradh, for a can of water.]","[{'code': 'SOM', 'place': 'Kerry'}]",0.90,[]


,entry_id,source_text,data,flags,needs_review
0,0,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...","{'headword_raw': 'a', 'headword': 'a', 'pronunciations': ['/ə/'], 'part_of_speech': 'indefinite article', 'definition': 'Indefinite article.', 'examples': ['I’ve had my ’nough now', 'She cried her...","{'missing_headword': False, 'missing_definition': False, 'bad_examples': False}",False
1,1,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...","{'headword_raw': 'a', 'headword': 'a', 'pronunciations': ['/ə/'], 'part_of_speech': 'voc. particle', 'definition': 'used when speaking to somebody', 'examples': ['‘A mhaoineach’, my dear one', '‘a...","{'missing_headword': False, 'missing_definition': False, 'bad_examples': False}",False



## 4. Flatten GenAI nested structure

This assumes the GenAI file stores nested `data` and `flags` objects.
If your structure differs, adjust this cell only.


In [47]:

def flatten_genai(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "data" in out.columns:
        data_flat = json_normalize(out["data"]).add_prefix("data.")
        out = pd.concat([out.drop(columns=["data"]), data_flat], axis=1)

    if "flags" in out.columns:
        flags_flat = json_normalize(out["flags"]).add_prefix("flags.")
        out = pd.concat([out.drop(columns=["flags"]), flags_flat], axis=1)

    return out


genai_flat = flatten_genai(genai_raw)

print("genai_flat shape:", genai_flat.shape)
display(genai_flat.head(2))


genai_flat shape: (3899, 15)


,entry_id,source_text,needs_review,data.headword_raw,data.headword,data.pronunciations,data.part_of_speech,data.definition,data.examples,data.etymology,data.cross_references,data.region_mentions,flags.missing_headword,flags.missing_definition,flags.bad_examples
0,1,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",False,a,a,[/ə/],indefinite article,Indefinite article.,"[I’ve had my ’nough now, She cried her ’nough.]","The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’ (cf. Ir Tá mo dhóthai...",[],[Mayo],False,False,False
1,2,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",False,a,a,[/ə/],voc. particle,used when speaking to somebody,"[‘A mhaoineach’, my dear one, ‘a Dhia’, O God, ‘a chairde’, friends, ‘a ghrá’, also ‘agradh’, my love, ‘a stór’, darling, ‘avourneen’, a term of endearment, “Arrah, Tim, avourneen, why did you die...","From Irish vocative particle 'a'. Includes references to Ir. 'A mhaoineach', 'a mhuirnín'. Dinneen notes: 'Eng. O, though not an equivalent, represents it.'","[agradh, a stór, avourneen]",[Kerry],False,False,False



## 5. Canonical field mapping

Keep this as the single source of truth for how raw columns map into the shared schema.


In [48]:

RULE_COLUMN_MAP = {
    "entry_id": "id",
    "source_text": "source_text",
    "headword_raw": "headword_raw",
    "headword": "headword",
    "variant_forms_raw": "variant_forms_raw",
    "variant_forms": "variant_forms",
    "definition": "definition",
    "etymology": "etymology",
    "examples": "examples",
    "cross_references": "cross_references",
    "region_mentions": "region_mentions",
    "needs_review": None,  # not present in rule output by default
}

GENAI_COLUMN_MAP = {
    "entry_id": "entry_id",
    "source_text": "source_text",
    "headword_raw": "data.headword_raw",
    "headword": "data.headword",
    "variant_forms_raw": "data.variant_forms_raw",
    "variant_forms": "data.variant_forms",
    "definition": "data.definition",
    "etymology": "data.etymology",
    "examples": "data.examples",
    "cross_references": "data.cross_references",
    "region_mentions": "data.region_mentions",
    "needs_review": "flags.needs_review",
}

RULE_PRONUNCIATION_COLUMNS = [
    "pronunciation",
    "pronunciation_2",
    "pronunciation_3",
    "pronunciation_4",
]

RULE_POS_COLUMN = "part_of_speech"
GENAI_POS_COLUMN = "data.part_of_speech"



## 6. Normalization functions

The goal is **comparison-safe normalization**, not rewriting the lexical data.


In [49]:

def clean_pronunciations(values: Any) -> list[str]:
    items = clean_list_text(values)
    out = []
    for item in items:
        item = item.strip().strip("/")
        item = clean_text(item)
        if item:
            out.append(item)
    return unique_keep_order(out)


def clean_pos(values: Any) -> list[str]:
    return unique_keep_order([x.casefold() for x in clean_list_text(values)])


def clean_cross_refs(values: Any) -> list[str]:
    return unique_keep_order([x.upper() for x in clean_list_text(values)])


def clean_examples(values: Any) -> list[str]:
    return clean_list_text(values)


def clean_region_values(values: Any) -> list[str]:
    cleaned = []
    for item in ensure_list(values):
        if isinstance(item, dict):
            for v in item.values():
                text = clean_text(v)
                if text:
                    cleaned.append(text)
        else:
            text = clean_text(item)
            if text:
                cleaned.append(text)
    return unique_keep_order(cleaned)


def build_rule_pronunciations(row: pd.Series) -> list[str]:
    values = [row.get(col) for col in RULE_PRONUNCIATION_COLUMNS]
    return clean_pronunciations(values)



## 7. Normalize rule-based output


In [50]:

def normalize_rule(df: pd.DataFrame) -> pd.DataFrame:
    out_rows = []

    for _, row in df.iterrows():
        record = {
            "entry_id": safe_int_from_id(row.get(RULE_COLUMN_MAP["entry_id"])),
            "source_text": clean_text(row.get(RULE_COLUMN_MAP["source_text"])),
            "headword_raw": clean_text(row.get(RULE_COLUMN_MAP["headword_raw"])),
            "headword": clean_text(row.get(RULE_COLUMN_MAP["headword"])),
            "variant_forms_raw": clean_text(row.get(RULE_COLUMN_MAP["variant_forms_raw"])),
            "variant_forms": clean_list_text(row.get(RULE_COLUMN_MAP["variant_forms"])),
            "pronunciations": build_rule_pronunciations(row),
            "part_of_speech": clean_pos(row.get(RULE_POS_COLUMN)),
            "definition": clean_text(row.get(RULE_COLUMN_MAP["definition"])),
            "examples": clean_examples(row.get(RULE_COLUMN_MAP["examples"])),
            "etymology": clean_text(row.get(RULE_COLUMN_MAP["etymology"])),
            "cross_references": clean_cross_refs(row.get(RULE_COLUMN_MAP["cross_references"])),
            "region_mentions": clean_region_values(row.get(RULE_COLUMN_MAP["region_mentions"])),
            "needs_review": False,
        }
        out_rows.append(record)

    out = pd.DataFrame(out_rows)
    return out[CORE_FIELDS].copy()


rule_norm = normalize_rule(rule_raw)
print(rule_norm.shape)
display(rule_norm.head(3))


(3898, 14)


,entry_id,source_text,headword_raw,headword,variant_forms_raw,variant_forms,pronunciations,part_of_speech,definition,examples,etymology,cross_references,region_mentions,needs_review
0,1,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",a,a,NaN,[],[ə],[],"indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’; ...",[She cried her ’nough.],NaN,[],"[PR, Mayo]",False
1,2,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",a,a,NaN,[],[ə],[voc.],"particle (used when speaking to somebody). “Arrah, Tim, avourneen, why did you die?” Joyce, Finnegans Wake, 468.34: “I’m dreaming of ye, azores [a stór]”; Healy, Nineteen Acres, 9: “Run down to th...","[Arrah, Tim, avourneen, why did you die?, I’m dreaming of ye, azores [a stór], Run down to the well, agradh, for a can of water.]","Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘avourneen’, a term of endearment < Ir a mhuirnín. ‘Finneg...",[],"[SOM, Kerry]",False
2,3,"ABCs /eːbiːsiːz/ n. (colloq.), irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN.",ABCs,abcs,NaN,[],[eːbiːsiːz],[n.],"irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN",[],NaN,[BRACKEN],"[JOM, Kerry]",False



## 8. Normalize GenAI output


In [51]:

def normalize_genai(df: pd.DataFrame) -> pd.DataFrame:
    out_rows = []

    for _, row in df.iterrows():
        record = {
            "entry_id": safe_int_from_id(row.get(GENAI_COLUMN_MAP["entry_id"])),
            "source_text": clean_text(row.get(GENAI_COLUMN_MAP["source_text"])),
            "headword_raw": clean_text(row.get(GENAI_COLUMN_MAP["headword_raw"])),
            "headword": clean_text(row.get(GENAI_COLUMN_MAP["headword"])),
            "variant_forms_raw": clean_text(row.get(GENAI_COLUMN_MAP["variant_forms_raw"])),
            "variant_forms": clean_list_text(row.get(GENAI_COLUMN_MAP["variant_forms"])),
            "pronunciations": clean_pronunciations(row.get("data.pronunciations")),
            "part_of_speech": clean_pos(row.get(GENAI_POS_COLUMN)),
            "definition": clean_text(row.get(GENAI_COLUMN_MAP["definition"])),
            "examples": clean_examples(row.get(GENAI_COLUMN_MAP["examples"])),
            "etymology": clean_text(row.get(GENAI_COLUMN_MAP["etymology"])),
            "cross_references": clean_cross_refs(row.get(GENAI_COLUMN_MAP["cross_references"])),
            "region_mentions": clean_region_values(row.get(GENAI_COLUMN_MAP["region_mentions"])),
            "needs_review": bool(row.get(GENAI_COLUMN_MAP["needs_review"])) if not is_nullish(row.get(GENAI_COLUMN_MAP["needs_review"])) else False,
        }
        out_rows.append(record)

    out = pd.DataFrame(out_rows)
    return out[CORE_FIELDS].copy()


genai_norm = normalize_genai(genai_flat)
print(genai_norm.shape)
display(genai_norm.head(3))


(3899, 14)


,entry_id,source_text,headword_raw,headword,variant_forms_raw,variant_forms,pronunciations,part_of_speech,definition,examples,etymology,cross_references,region_mentions,needs_review
0,1,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",a,a,None,[],[ə],[indefinite article],Indefinite article.,"[I’ve had my ’nough now, She cried her ’nough.]","The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’ (cf. Ir Tá mo dhóthai...",[],[Mayo],False
1,2,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",a,a,None,[],[ə],[voc. particle],used when speaking to somebody,"[‘A mhaoineach’, my dear one, ‘a Dhia’, O God, ‘a chairde’, friends, ‘a ghrá’, also ‘agradh’, my love, ‘a stór’, darling, ‘avourneen’, a term of endearment, “Arrah, Tim, avourneen, why did you die...","From Irish vocative particle 'a'. Includes references to Ir. 'A mhaoineach', 'a mhuirnín'. Dinneen notes: 'Eng. O, though not an equivalent, represents it.'","[AGRADH, A STÓR, AVOURNEEN]",[Kerry],False
2,3,"ABCs /eːbiːsiːz/ n. (colloq.), irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN.",ABCs,ABCs,None,[],[eːbiːsiːz],[n.],irregular red lines on children’s and women’s shins caused by sitting too close to open fires,[],NaN,[BRACKEN],[Kerry],False



## 9. Basic quality checks


In [52]:

def empty_count(series: pd.Series) -> int:
    return int(series.apply(is_nullish).sum())


qc = pd.DataFrame({
    "column": CORE_FIELDS,
    "rule_empty": [empty_count(rule_norm[c]) for c in CORE_FIELDS],
    "genai_empty": [empty_count(genai_norm[c]) for c in CORE_FIELDS],
})

display(qc)


,column,rule_empty,genai_empty
0,entry_id,0,0
1,source_text,0,0
2,headword_raw,0,0
3,headword,0,0
4,variant_forms_raw,3509,3899
5,variant_forms,3510,3899
6,pronunciations,583,539
7,part_of_speech,693,462
8,definition,367,319
9,examples,3124,1208



## 10. Optional normalization maps for POS and regions

Use JSON maps if you want controlled canonical labels.
If the files do not exist, the notebook will skip this step gracefully.


In [53]:

def load_json_if_exists(path: Path) -> dict[str, Any]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}


POS_MAP_PATH = PROJECT_DIR / "Resources" / "pos_map.json"
REGION_MAP_PATH = PROJECT_DIR / "Resources" / "region_map.json"


POS_MAP = load_json_if_exists(POS_MAP_PATH)
REGION_MAP = load_json_if_exists(REGION_MAP_PATH)

print("Loaded POS map:", bool(POS_MAP))
print("Loaded REGION map:", bool(REGION_MAP))


Loaded POS map: True
Loaded REGION map: True


In [54]:

def apply_pos_map(values: list[str], pos_map: dict[str, str]) -> list[str]:
    if not pos_map:
        return values
    mapped = [pos_map.get(v, v) for v in values]
    return unique_keep_order(mapped)


def apply_region_map(values: list[str], region_map: dict[str, str]) -> list[str]:
    if not region_map:
        return values
    mapped = [region_map.get(v, v) for v in values]
    return unique_keep_order(mapped)


for df in (rule_norm, genai_norm):
    df["part_of_speech"] = df["part_of_speech"].apply(lambda x: apply_pos_map(x, POS_MAP))
    df["region_mentions"] = df["region_mentions"].apply(lambda x: apply_region_map(x, REGION_MAP))



## 11. Merge datasets

Always merge on the normalized integer `entry_id`.


In [55]:

merged = rule_norm.merge(
    genai_norm,
    on="entry_id",
    how="outer",
    suffixes=("_rule", "_genai"),
    indicator=True,
)

print(merged["_merge"].value_counts(dropna=False))
display(merged.head(3))


_merge
both          3898
right_only       1
left_only        0
Name: count, dtype: int64


,entry_id,source_text_rule,headword_raw_rule,headword_rule,variant_forms_raw_rule,variant_forms_rule,pronunciations_rule,part_of_speech_rule,definition_rule,examples_rule,etymology_rule,cross_references_rule,region_mentions_rule,needs_review_rule,source_text_genai,headword_raw_genai,headword_genai,variant_forms_raw_genai,variant_forms_genai,pronunciations_genai,part_of_speech_genai,definition_genai,examples_genai,etymology_genai,cross_references_genai,region_mentions_genai,needs_review_genai,_merge
0,1,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",a,a,NaN,[],[ə],[],"indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’; ...",[She cried her ’nough.],NaN,[],"[PR, Mayo]",False,"a /ə/ indefinite article. The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NO...",a,a,None,[],[ə],[indefinite_article],Indefinite article.,"[I’ve had my ’nough now, She cried her ’nough.]","The absence of an indefinite article in Irish sometimes led to confusion with the initial sound in such English words as ‘enough’, giving rise to formations such as ‘a NOUGH’ (cf. Ir Tá mo dhóthai...",[],[Mayo],False,both
1,2,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",a,a,NaN,[],[ə],[vocative],"particle (used when speaking to somebody). “Arrah, Tim, avourneen, why did you die?” Joyce, Finnegans Wake, 468.34: “I’m dreaming of ye, azores [a stór]”; Healy, Nineteen Acres, 9: “Run down to th...","[Arrah, Tim, avourneen, why did you die?, I’m dreaming of ye, azores [a stór], Run down to the well, agradh, for a can of water.]","Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘avourneen’, a term of endearment < Ir a mhuirnín. ‘Finneg...",[],"[SOM, Kerry]",False,"a /ə/ voc. particle (used when speaking to somebody) < Ir. ‘A mhaoineach’, my dear one; ‘a Dhia’, O God; ‘a chairde’, friends; ‘a ghrá’, also ‘agradh’, my love; ‘a stór’, darling (SOM, Kerry); ‘av...",a,a,None,[],[ə],[vocative],used when speaking to somebody,"[‘A mhaoineach’, my dear one, ‘a Dhia’, O God, ‘a chairde’, friends, ‘a ghrá’, also ‘agradh’, my love, ‘a stór’, darling, ‘avourneen’, a term of endearment, “Arrah, Tim, avourneen, why did you die...","From Irish vocative particle 'a'. Includes references to Ir. 'A mhaoineach', 'a mhuirnín'. Dinneen notes: 'Eng. O, though not an equivalent, represents it.'","[AGRADH, A STÓR, AVOURNEEN]",[Kerry],False,both
2,3,"ABCs /eːbiːsiːz/ n. (colloq.), irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN.",ABCs,abcs,NaN,[],[eːbiːsiːz],[noun],"irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN",[],NaN,[BRACKEN],"[JOM, Kerry]",False,"ABCs /eːbiːsiːz/ n. (colloq.), irregular red lines on children’s and women’s shins caused by sitting too close to open fires (JOM, Kerry). See BRACKEN.",ABCs,ABCs,None,[],[eːbiːsiːz],[noun],irregular red lines on children’s and women’s shins caused by sitting too close to open fires,[],NaN,[BRACKEN],[Kerry],False,both



## 12. Alignment validation

This is a **cannot-change** step.
Check whether merged rows actually correspond to the same source entry.


In [56]:

merged["source_text_exact_match"] = merged.apply(
    lambda r: exact_text_match(r.get("source_text_rule"), r.get("source_text_genai")),
    axis=1,
)

alignment_summary = (
    merged.groupby("_merge")["source_text_exact_match"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"sum": "exact_match_count", "mean": "exact_match_rate"})
)

display(alignment_summary)

misaligned = merged[
    (merged["_merge"] == "both") &
    (~merged["source_text_exact_match"])
].copy()

print("Potentially misaligned rows:", len(misaligned))
display(misaligned[["entry_id", "source_text_rule", "source_text_genai"]].head(10))


,_merge,count,exact_match_count,exact_match_rate
0,right_only,1,0,0.000000
1,both,3898,3701,0.949461


Potentially misaligned rows: 197


,entry_id,source_text_rule,source_text_genai
8,9,"aboo /əˈbuː/ int., for ever! to victory! < Ir abú. Banim, The Boyne Water, 117: “‘Rhia Shamus Abo!’ cried the man, rising his cup... ‘King Shamus!’ repeated another, translating his friend’s Irish...","aboo /əˈbuː/ int., for ever! to victory! < Ir abú. Banim, The Boyne Water, 117: “‘Rhia Shamus Abo!’ cried the man, rising his cup . . . ‘King Shamus!’ repeated another, translating his friend’s Ir..."
13,14,"a chara /əˈxɑrə/ voc., term of endearment: my friend < Ir. ‘Mhuise, a chara, come out for a walk with me and don’t be moping around the house all day.’ Kickham, Knocknagow, 57: “Billy, a chora... ...","a chara /əˈxɑrə/ voc., term of endearment: my friend < Ir. ‘Mhuise, a chara, come out for a walk with me and don’t be moping around the house all day.’ Kickham, Knocknagow, 57: “Billy, a chora ......"
30,31,"after /ˈæːftər/, /ˈæːftər/ prep., used to form the HE equivalent of the SE perfect (have...) and pluperfect (had...) past tenses (see Filppula, Grammar, 1999, 99–107). There is no verb ‘have’ in I...","after /ˈæːftər/, /ˈæːftər/ prep., used to form the HE equivalent of the SE perfect (have ...) and pluperfect (had...) past tenses (see Filppula, Grammar, 1999, 99–107). There is no verb ‘have’ in ..."
46,47,"ail /eːl/ v., to be amiss with, to be afflicted < E dial. (archaic in SE) < OE eglan, to trouble. ‘Teil v., to ail, to be amiss, “Fade teil?” What ails?’ (JP, Forth and Bargy); ‘What ails you? You...","ail /eːl/ v., to be amiss with, to be afflicted < E dial. (archaic in SE) < OE eglan, to trouble. ‘Teil v., to ail, to be amiss, “Fade teil?” What ails?’ (JP, Forth and Bargy); ‘What ails you? You..."
94,95,"and /ænd/ conj. This word has a much wider range of use in HE than in SE, because in Irish the conjunction agus (and) commonly functions as a subordinating adverbial conjunction (e.g. ‘when’, ‘bec...","and /ænd/ conj. This word has a much wider range of use in HE than in SE, because in Irish the conjunction agus (and) commonly functions as a subordinating adverbial conjunction (e.g. ‘when’, ‘bec..."
100,101,"Angelus /ˈænʤələs/ n., a prayer recited in honour of the Incarnation, taken from the first word, ‘The Angel [Angelus] of the Lord declared unto Mary...’ < L angelus, Gr aggelos, messenger. The pra...","Angelus /ˈænʤələs/ n., a prayer recited in honour of the Incarnation, taken from the first word, ‘The Angel [Angelus] of the Lord declared unto Mary ...’ < L angelus, Gr aggelos, messenger. The pr..."
124,125,"aroon /əˈruːn/ int., term of endearment: dear, dear man, dear woman < Ir a rún, loved one (with voc. particle). ‘Aroon, you haven’t a clue what you’re talking about.’ Joyce, Finnegans Wake, 613.34...","aroon /əˈruːn/ int., term of endearment: dear, dear man, dear woman < Ir a rún, loved one (with voc. particle). ‘Aroon, you haven’t a clue what you’re talking about.’ Joyce, Finnegans Wake, 613.34..."
125,126,"arrah /ˈærə/ also ara, arra, arú int., now, but, really, ‘phrase to indicate that a situation is not to be taken too seriously’ (SOM, Kerry, who adds: ‘younger people use “YERRAH”’ < Ir dhera). ‘A...","arrah /ˈærə/ also ara, arra, arú int., now, but, really, ‘phrase to indicate that a situation is not to be taken too seriously’ (SOM, Kerry, who adds: ‘younger people use “YERRAH”’ < Ir dhera). ‘A..."
134,135,"aself /əˈsɛlf/ itself; even. O’Casey, The Plough and the Stars, act II, 175: “Rosie:... an’ if I’m a prostitute aself [even if I’m a prostitute], I have me feelin’s.”","aself /əˈsɛlf/ itself; even. O’Casey, The Plough and the Stars, act II, 175: “Rosie: . . . an’ if I’m a prostitute aself [even if I’m a prostitute], I have me feelin’s.”"
203,204,"ballyrag /ˈbæliræɡ/ v., chide, scold < E dial. (origin obscure). Trollope, The Kellys and the O’Kellys, 236: “But av’ [if] he’s nothing betther for you to do, than to send you here bally-ragging...”","ballyrag /ˈbæliræɡ/ v., chide, scold < E dial. (origin obscure). Trollope, The Kellys and the O’Kellys, 236: “Bu


## 13. Presence / absence comparison


In [57]:

COMPARE_FIELDS = [
    "headword_raw",
    "headword",
    "variant_forms_raw",
    "variant_forms",
    "pronunciations",
    "part_of_speech",
    "definition",
    "examples",
    "etymology",
    "cross_references",
    "region_mentions",
]


def presence_status(rule_value: Any, genai_value: Any) -> str:
    r = not is_nullish(rule_value)
    g = not is_nullish(genai_value)
    if r and g:
        return "both_present"
    if r and not g:
        return "rule_only"
    if not r and g:
        return "genai_only"
    return "both_missing"


presence_rows = []
for field in COMPARE_FIELDS:
    status_col = f"{field}_presence_status"
    merged[status_col] = merged.apply(
        lambda r: presence_status(r.get(f"{field}_rule"), r.get(f"{field}_genai")),
        axis=1,
    )
    summary = merged[status_col].value_counts(dropna=False).to_dict()
    summary["field"] = field
    presence_rows.append(summary)

presence_summary = pd.DataFrame(presence_rows).fillna(0)
presence_summary = presence_summary[
    ["field", "both_present", "rule_only", "genai_only", "both_missing"]
].sort_values("field")

display(presence_summary)


,field,both_present,rule_only,genai_only,both_missing
9,cross_references,633.0,2.0,777.0,2487.0
6,definition,3493.0,38.0,87.0,281.0
8,etymology,2692.0,1.0,351.0,855.0
7,examples,749.0,25.0,1942.0,1183.0
1,headword,3898.0,0.0,1.0,0.0
0,headword_raw,3898.0,0.0,1.0,0.0
5,part_of_speech,3205.0,0.0,232.0,462.0
4,pronunciations,3315.0,0.0,45.0,539.0
10,region_mentions,2097.0,0.0,857.0,945.0
3,variant_forms,0.0,388.0,0.0,3511.0



## 14. Exact match comparison


In [58]:

def field_exact_match(field: str, row: pd.Series) -> bool:
    left = row.get(f"{field}_rule")
    right = row.get(f"{field}_genai")

    if field in TEXT_FIELDS:
        return exact_text_match(left, right)
    return exact_list_match(left, right)


for field in COMPARE_FIELDS:
    merged[f"{field}_exact_match"] = merged.apply(lambda r, f=field: field_exact_match(f, r), axis=1)

exact_match_summary = pd.DataFrame({
    "field": COMPARE_FIELDS,
    "exact_match_rate": [merged[f"{f}_exact_match"].mean() for f in COMPARE_FIELDS],
})

display(exact_match_summary.sort_values("exact_match_rate", ascending=False))


,field,exact_match_rate
4,pronunciations,0.952552
1,headword,0.918954
3,variant_forms,0.900487
2,variant_forms_raw,0.900231
0,headword_raw,0.886125
5,part_of_speech,0.873814
9,cross_references,0.779174
10,region_mentions,0.377533
7,examples,0.317774
8,etymology,0.251090



## 15. Containment diagnostics

Evaluate how well extracted content is grounded in the source using consumptive matching.

We use three complementary approaches:

Phrase containment (strict)
Checks if extracted strings appear in the source text
Good for detecting exact grounding
Sensitive to rephrasing and segmentation
Unordered token containment (consumptive)
Token-level matching using a multiset (bag of words)
Each matched token is consumed once (no duplicate reuse)
Handles reordering and segmented extraction

Gives:

precision → how much extracted text is grounded
recall → how much of the source is captured
Ordered token containment (consumptive)
Token-level matching left-to-right
Each match consumes one occurrence and advances position
Penalizes reordering

Gives a stricter measure of extraction fidelity

Notes
Consumptive matching prevents duplicate tokens from being overcounted
Phrase = strict grounding
Unordered = flexible lexical overlap
Ordered = strict sequence fidelity

In [59]:
#tokenization helper

def tokenize_for_matching(text: str) -> list[str]:
    if not text:
        return []
    text = normalize_text_for_match(text)
    return re.findall(r"\b\w+\b", text)




#Unordered consumptive matching

#This is multiset-style matching with one-token-at-a-time consumption.

def unordered_consumptive_match(source_value: Any, output_value: Any) -> dict[str, Any]:
    source_tokens = []
    for s in to_text_list(source_value):
        source_tokens.extend(tokenize_for_matching(s))

    output_tokens = []
    for o in to_text_list(output_value):
        output_tokens.extend(tokenize_for_matching(o))

    source_counter = Counter(source_tokens)
    output_counter = Counter(output_tokens)

    matched_counter = Counter()
    for tok, out_count in output_counter.items():
        matched_counter[tok] = min(out_count, source_counter.get(tok, 0))

    matched_total = sum(matched_counter.values())
    source_total = sum(source_counter.values())
    output_total = sum(output_counter.values())

    extra_counter = output_counter - matched_counter
    missing_counter = source_counter - matched_counter

    precision = matched_total / output_total if output_total else None
    recall = matched_total / source_total if source_total else None
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None and recall is not None and (precision + recall) > 0
        else None
    )

    return {
        "matched_tokens": matched_total,
        "output_tokens": output_total,
        "source_tokens": source_total,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "extra_tokens": dict(extra_counter),
        "missing_tokens": dict(missing_counter),
    }



#4. Ordered consumptive matching

#This is stricter and keeps sequence.

def ordered_consumptive_match(source_value: Any, output_value: Any) -> dict[str, Any]:
    source_tokens = []
    for s in to_text_list(source_value):
        source_tokens.extend(tokenize_for_matching(s))

    output_tokens = []
    for o in to_text_list(output_value):
        output_tokens.extend(tokenize_for_matching(o))

    source_used = [False] * len(source_tokens)
    matched_positions = []
    unmatched_output = []

    last_match_idx = -1
    matched_total = 0

    for tok in output_tokens:
        found_idx = None

        for i in range(last_match_idx + 1, len(source_tokens)):
            if not source_used[i] and source_tokens[i] == tok:
                found_idx = i
                break

        if found_idx is not None:
            source_used[found_idx] = True
            matched_positions.append(found_idx)
            last_match_idx = found_idx
            matched_total += 1
        else:
            unmatched_output.append(tok)

    missing_source = [tok for tok, used in zip(source_tokens, source_used) if not used]

    output_total = len(output_tokens)
    source_total = len(source_tokens)

    precision = matched_total / output_total if output_total else None
    recall = matched_total / source_total if source_total else None
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None and recall is not None and (precision + recall) > 0
        else None
    )

    return {
        "matched_tokens": matched_total,
        "output_tokens": output_total,
        "source_tokens": source_total,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "unmatched_output_tokens": unmatched_output,
        "missing_source_tokens": missing_source,
        "matched_source_positions": matched_positions,
    }

#5. Run both across your containment fields

#Add a new section after your phrase containment block:


TOKEN_CONTAINMENT_FIELDS = [
    "headword_raw",
    "headword",
    "variant_forms",
    "pronunciations",
    "part_of_speech",
    "definition",
    "examples",
    "etymology",
    "cross_references",
]

token_match_rows = []

for field in TOKEN_CONTAINMENT_FIELDS:
    for system in ("rule", "genai"):
        source_col = "source_text_rule"

        unordered_col = f"{field}_{system}_unordered_token_match"
        ordered_col = f"{field}_{system}_ordered_token_match"

        merged[unordered_col] = merged.apply(
            lambda r, f=field, s=system: unordered_consumptive_match(
                r.get(source_col) if not is_nullish(r.get(source_col)) else r.get("source_text_genai"),
                r.get(f"{f}_{s}")
            ),
            axis=1,
        )

        merged[ordered_col] = merged.apply(
            lambda r, f=field, s=system: ordered_consumptive_match(
                r.get(source_col) if not is_nullish(r.get(source_col)) else r.get("source_text_genai"),
                r.get(f"{f}_{s}")
            ),
            axis=1,
        )

        token_match_rows.append({
            "field": field,
            "system": system,
            "unordered_precision": merged[unordered_col].apply(lambda x: x["precision"]).mean(),
            "unordered_recall": merged[unordered_col].apply(lambda x: x["recall"]).mean(),
            "unordered_f1": merged[unordered_col].apply(lambda x: x["f1"]).mean(),
            "ordered_precision": merged[ordered_col].apply(lambda x: x["precision"]).mean(),
            "ordered_recall": merged[ordered_col].apply(lambda x: x["recall"]).mean(),
            "ordered_f1": merged[ordered_col].apply(lambda x: x["f1"]).mean(),
        })

token_match_summary = pd.DataFrame(token_match_rows)
display(token_match_summary.sort_values(["field", "system"]))

token_match_summary.to_excel(writer, sheet_name="token_match_summary", index=False)


def make_json_serializable(value):
    if isinstance(value, dict):
        return {k: make_json_serializable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [make_json_serializable(v) for v in value]
    if isinstance(value, tuple):
        return [make_json_serializable(v) for v in value]
    if pd.isna(value) and not isinstance(value, (str, list, dict)):
        return None
    return value


def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    records = df.to_dict(orient="records")
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            clean_record = {k: make_json_serializable(v) for k, v in record.items()}
            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")





,field,system,unordered_precision,unordered_recall,unordered_f1,ordered_precision,ordered_recall,ordered_f1
17,cross_references,genai,0.998475,0.048482,0.210915,0.996879,0.048433,0.210677
16,cross_references,rule,0.941732,0.033930,0.332774,0.940420,0.033927,0.332728
11,definition,genai,0.992589,0.256714,0.401150,0.982225,0.253630,0.396419
10,definition,rule,1.000000,0.454030,0.617470,1.000000,0.454030,0.617470
15,etymology,genai,0.947184,0.104971,0.219918,0.941946,0.104175,0.218288
14,etymology,rule,1.000000,0.325596,0.601448,1.000000,0.325596,0.601448
13,examples,genai,0.999286,0.331408,0.627261,0.997761,0.330736,0.626091
12,examples,rule,1.000000,0.057258,0.422090,1.000000,0.057258,0.422090
3,headword,genai,0.984440,0.074908,0.128371,0.982439,0.074479,0.127657
2,headword,rule,1.000000,0.077440,0.130653,1.000000,0.077440,0.130653



## 16. Length-based overgeneration / omission diagnostics

These are **heuristics**, not proof of hallucination.


In [60]:

for suffix in ("rule", "genai"):
    merged[f"extracted_char_len_{suffix}"] = merged.apply(
        lambda r, s=suffix: sum(
            char_len(r.get(f"{field}_{s}")) for field in COMPARE_FIELDS
        ),
        axis=1,
    )
    merged[f"extracted_word_len_{suffix}"] = merged.apply(
        lambda r, s=suffix: sum(
            word_len(r.get(f"{field}_{s}")) for field in COMPARE_FIELDS
        ),
        axis=1,
    )

merged["source_char_len"] = merged["source_text_rule"].fillna(merged["source_text_genai"]).apply(char_len)
merged["source_word_len"] = merged["source_text_rule"].fillna(merged["source_text_genai"]).apply(word_len)

for suffix in ("rule", "genai"):
    merged[f"char_ratio_{suffix}"] = merged[f"extracted_char_len_{suffix}"] / merged["source_char_len"].replace(0, np.nan)
    merged[f"word_ratio_{suffix}"] = merged[f"extracted_word_len_{suffix}"] / merged["source_word_len"].replace(0, np.nan)
    merged[f"char_diff_{suffix}"] = merged[f"extracted_char_len_{suffix}"] - merged["source_char_len"]
    merged[f"word_diff_{suffix}"] = merged[f"extracted_word_len_{suffix}"] - merged["source_word_len"]

    merged[f"possible_overgeneration_{suffix}"] = merged[f"char_ratio_{suffix}"] > 1.20
    merged[f"possible_omission_{suffix}"] = merged[f"word_ratio_{suffix}"] < 0.80

length_summary = pd.DataFrame({
    "metric": [
        "possible_overgeneration_rule",
        "possible_overgeneration_genai",
        "possible_omission_rule",
        "possible_omission_genai",
    ],
    "rate": [
        merged["possible_overgeneration_rule"].mean(),
        merged["possible_overgeneration_genai"].mean(),
        merged["possible_omission_rule"].mean(),
        merged["possible_omission_genai"].mean(),
    ]
})

display(length_summary)


,metric,rate
0,possible_overgeneration_rule,0.160298
1,possible_overgeneration_genai,0.058989
2,possible_omission_rule,0.000513
3,possible_omission_genai,0.049500



## 17. Failure extracts for manual review


In [62]:
failure_tables: dict[str, pd.DataFrame] = {}

for field in CONTAINMENT_FIELDS:
    for system in ("rule", "genai"):

        # -----------------------------
        # 1. Phrase containment failures
        # -----------------------------
        phrase_col = f"{field}_{system}_containment"

        if phrase_col in merged.columns:
            subset = merged[
                merged[phrase_col].apply(lambda x: not x.all_output_in_source)
            ][[
                "entry_id",
                "source_text_rule",
                "source_text_genai",
                f"{field}_{system}",
                phrase_col,
            ]].copy()

            subset["unmatched_output_items"] = subset[phrase_col].apply(lambda x: x.unmatched_output_items)
            subset["output_coverage_in_source"] = subset[phrase_col].apply(lambda x: x.output_coverage_in_source)

            failure_tables[f"{field}_{system}_phrase"] = subset.drop(columns=[phrase_col])


        # -----------------------------
        # 2. Unordered token failures
        # -----------------------------
        unordered_col = f"{field}_{system}_unordered_token_match"

        if unordered_col in merged.columns:
            subset = merged[
                merged[unordered_col].apply(lambda x: x["precision"] is not None and x["precision"] < 1.0)
            ][[
                "entry_id",
                "source_text_rule",
                "source_text_genai",
                f"{field}_{system}",
                unordered_col,
            ]].copy()

            subset["precision"] = subset[unordered_col].apply(lambda x: x["precision"])
            subset["recall"] = subset[unordered_col].apply(lambda x: x["recall"])
            subset["extra_tokens"] = subset[unordered_col].apply(lambda x: x["extra_tokens"])
            subset["missing_tokens"] = subset[unordered_col].apply(lambda x: x["missing_tokens"])

            failure_tables[f"{field}_{system}_unordered"] = subset.drop(columns=[unordered_col])


        # -----------------------------
        # 3. Ordered token failures
        # -----------------------------
        ordered_col = f"{field}_{system}_ordered_token_match"

        if ordered_col in merged.columns:
            subset = merged[
                merged[ordered_col].apply(lambda x: x["precision"] is not None and x["precision"] < 1.0)
            ][[
                "entry_id",
                "source_text_rule",
                "source_text_genai",
                f"{field}_{system}",
                ordered_col,
            ]].copy()

            subset["precision"] = subset[ordered_col].apply(lambda x: x["precision"])
            subset["recall"] = subset[ordered_col].apply(lambda x: x["recall"])
            subset["unmatched_output_tokens"] = subset[ordered_col].apply(lambda x: x["unmatched_output_tokens"])
            subset["missing_source_tokens"] = subset[ordered_col].apply(lambda x: x["missing_source_tokens"])

            failure_tables[f"{field}_{system}_ordered"] = subset.drop(columns=[ordered_col])


print("Prepared failure tables:", len(failure_tables))
list(failure_tables.keys())[:10]

Prepared failure tables: 36


['headword_raw_rule_unordered',
 'headword_raw_rule_ordered',
 'headword_raw_genai_unordered',
 'headword_raw_genai_ordered',
 'headword_rule_unordered',
 'headword_rule_ordered',
 'headword_genai_unordered',
 'headword_genai_ordered',
 'variant_forms_rule_unordered',
 'variant_forms_rule_ordered']


## 18. Export outputs


In [63]:
# JSON export (with safe serialization if needed)
write_jsonl(merged, MERGED_JSONL_PATH)

with pd.ExcelWriter(MERGED_XLSX_PATH, engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="merged", index=False)
    presence_summary.to_excel(writer, sheet_name="presence_summary", index=False)
    exact_match_summary.to_excel(writer, sheet_name="exact_match_summary", index=False)
    containment_summary.to_excel(writer, sheet_name="containment_summary", index=False)
    length_summary.to_excel(writer, sheet_name="length_summary", index=False)

presence_summary.to_excel(PRESENCE_SUMMARY_PATH, index=False)

# Failure reports (now richer)
with pd.ExcelWriter(CONTAINMENT_REPORT_PATH, engine="openpyxl") as writer:
    for sheet_name, df_ in failure_tables.items():
        safe_name = sheet_name[:31]  # Excel limit
        df_.to_excel(writer, sheet_name=safe_name, index=False)

print("Saved:")
print("-", MERGED_JSONL_PATH)
print("-", MERGED_XLSX_PATH)
print("-", PRESENCE_SUMMARY_PATH)
print("-", CONTAINMENT_REPORT_PATH)

Saved:
- D:\Experiments\Hiberno_English_Dictonary\hiberno_lexical_Eval_V2\outputs\final_merged_normalized.jsonl
- D:\Experiments\Hiberno_English_Dictonary\hiberno_lexical_Eval_V2\outputs\final_merged_normalized.xlsx
- D:\Experiments\Hiberno_English_Dictonary\hiberno_lexical_Eval_V2\outputs\presence_summary.xlsx
- D:\Experiments\Hiberno_English_Dictonary\hiberno_lexical_Eval_V2\outputs\containment_failures.xlsx



## 19. Recommended final workflow

1. Load raw data  
2. Flatten nested GenAI structure  
3. Normalize both systems into one canonical schema  
4. Normalize IDs before merge  
5. Outer merge  
6. Validate alignment with source text  
7. Compare presence / absence  
8. Compare exact normalized matches  
9. Run bidirectional containment  
10. Run length-based heuristics  
11. Export merged dataset and failure reports  

This keeps the notebook:
- readable
- reproducible
- defendable in a methods section
